#### Load dataset

In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("dataset/1_Daily_minimum_temps.csv")   # columns: Date, Temp

df["Temp"] = df["Temp"].astype(str).str.replace("?", "", regex=False)
df["Temp"] = df["Temp"].astype("float32")

temps = df["Temp"].values
temps

array([20.7, 17.9, 18.8, ..., 13.5, 15.7, 13. ],
      shape=(3650,), dtype=float32)

#### Normalize the data

In [12]:
scaler = MinMaxScaler()
temps_scaled = scaler.fit_transform(temps.reshape(-1, 1)).flatten()


In [16]:
temps_scaled

array([0.7870723 , 0.6806084 , 0.7148289 , ..., 0.51330805, 0.5969582 ,
       0.4942966 ], shape=(3650,), dtype=float32)

#### Create sequences (use last 12 days → predict next day)

In [13]:
SEQ_LEN = 12

X_list = []
y_list = []

for i in range(len(temps_scaled) - SEQ_LEN):
    X_list.append(temps_scaled[i:i+SEQ_LEN])
    y_list.append(temps_scaled[i+SEQ_LEN])

X = np.array(X_list).reshape(-1, SEQ_LEN, 1)
y = np.array(y_list).reshape(-1, 1)


In [14]:
X

array([[[0.7870723 ],
        [0.6806084 ],
        [0.7148289 ],
        ...,
        [0.7604563 ],
        [0.61596966],
        [0.50570345]],

       [[0.6806084 ],
        [0.7148289 ],
        [0.5551331 ],
        ...,
        [0.61596966],
        [0.50570345],
        [0.63498104]],

       [[0.7148289 ],
        [0.5551331 ],
        [0.6007605 ],
        ...,
        [0.50570345],
        [0.63498104],
        [0.8174906 ]],

       ...,

       [[0.5285171 ],
        [0.6539925 ],
        [0.5589354 ],
        ...,
        [0.5551331 ],
        [0.5323194 ],
        [0.5171103 ]],

       [[0.6539925 ],
        [0.5589354 ],
        [0.5855514 ],
        ...,
        [0.5323194 ],
        [0.5171103 ],
        [0.51330805]],

       [[0.5589354 ],
        [0.5855514 ],
        [0.4980989 ],
        ...,
        [0.5171103 ],
        [0.51330805],
        [0.5969582 ]]], shape=(3638, 12, 1), dtype=float32)

In [15]:
y

array([[0.63498104],
       [0.8174906 ],
       [0.9505704 ],
       ...,
       [0.51330805],
       [0.5969582 ],
       [0.4942966 ]], shape=(3638, 1), dtype=float32)

#### Create 3‑way chronological split (70/15/15)

In [17]:
n = len(X)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]


#### Bidirectional RNN Model (you can swap this block for any RNN type)

In [7]:
model = tf.keras.Sequential([
    tf.keras.layers.Bidirectional(
        tf.keras.layers.SimpleRNN(64, return_sequences=False)
    ),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### Train the model

In [8]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)


Epoch 1/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.0128 - mae: 0.0871 - val_loss: 0.0087 - val_mae: 0.0745
Epoch 2/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0094 - mae: 0.0759 - val_loss: 0.0099 - val_mae: 0.0801
Epoch 3/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0093 - mae: 0.0757 - val_loss: 0.0085 - val_mae: 0.0722
Epoch 4/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0097 - mae: 0.0772 - val_loss: 0.0080 - val_mae: 0.0705
Epoch 5/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0092 - mae: 0.0749 - val_loss: 0.0089 - val_mae: 0.0754
Epoch 6/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0092 - mae: 0.0753 - val_loss: 0.0082 - val_mae: 0.0707
Epoch 7/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0090 - mae: 0.0746 - val_loss: 0.0083 - val_mae: 0.0717
Epoch 8/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0091 - mae: 0.0748 - val_loss: 0.0083 - val_mae: 0.0723
Epoch 9/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0090 - mae:

#### Evaluate on the test set and train

In [10]:
train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)


Train MAE: 0.07171665877103806
Test MAE: 0.06740844994783401
